# Stage 5: Boosting and k-nearest neighbours

This notebook compares a gradient-boosting model with k-nearest neighbours using the shared feature and preprocessing pipeline. All comparisons use the training split; the final test set remains untouched.

In [8]:
from pathlib import Path
import json
import sys

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier

PROJECT_ROOT = Path.cwd().parents[0] if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data import get_split
from src.evaluate import cv_report, save_result
from src.pipeline import build

REPORTS_DIR = PROJECT_ROOT / 'reports'
BEST_PARAMS_PATH = REPORTS_DIR / 'best_params.json'
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

## Split first

In [9]:
X_train, X_test, y_train, y_test = get_split()
print(f'Training rows: {len(X_train):,}')
print(f'Held-out test rows: {len(X_test):,}')
print(f'Training yes-rate: {y_train.mean():.3%}')

Training rows: 36,168
Held-out test rows: 9,043
Training yes-rate: 11.698%


### Observation and decision
The split comes before cross-validation and tuning. Every transformer in `build()` is fitted inside each training fold, and the held-out test rows are not used for model selection.

## HistGradientBoosting baseline and tuning

In [10]:
# HistGradientBoosting is sklearn's fast implementation of the GBM idea from lectures.
hgb_baseline = build(HistGradientBoostingClassifier(class_weight='balanced', random_state=42))
baseline_row = cv_report('HistGradientBoosting', hgb_baseline, X_train, y_train)
save_result(baseline_row, REPORTS_DIR / 'model_comparison.csv')
baseline_row

{'name': 'HistGradientBoosting',
 'pr_auc_mean': np.float64(0.44018308503680104),
 'pr_auc_std': np.float64(0.00953980096325405),
 'roc_auc_mean': np.float64(0.7951364280236309),
 'roc_auc_std': np.float64(0.0027595737870123958),
 'f1_mean': np.float64(0.44586406900565534),
 'f1_std': np.float64(0.009917654521236146),
 'precision_mean': np.float64(0.3507823608010944),
 'precision_std': np.float64(0.01201596383975227),
 'recall_mean': np.float64(0.6121513560585128),
 'recall_std': np.float64(0.006734125666640066),
 'train_pr_auc_mean': np.float64(0.5224076890492517),
 'train_pr_auc_std': np.float64(0.009359372703182727),
 'fit_time_mean': np.float64(0.7182952404022217)}

### Observation and decision
Boosting grows trees in sequence. Each later tree focuses more on examples that the earlier trees predicted poorly, so the combined model gradually corrects earlier errors. The balanced class weight gives the minority yes class more influence during the baseline comparison.

In [11]:
hgb_search = RandomizedSearchCV(
    estimator=build(HistGradientBoostingClassifier(class_weight='balanced', random_state=42)),
    param_distributions={
        'model__learning_rate': [0.02, 0.05, 0.1, 0.2],
        'model__max_depth': [None, 3, 5, 7],
        'model__max_leaf_nodes': [15, 31, 63, 127],
        'model__min_samples_leaf': [10, 20, 50, 100],
        'model__l2_regularization': [0.0, 0.1, 1.0, 10.0],
    },
    n_iter=25,
    scoring='average_precision',
    cv=cv,
    random_state=42,
    n_jobs=-1,
)
hgb_search.fit(X_train, y_train)
hgb_tuned_row = cv_report('HistGradientBoosting-tuned', hgb_search.best_estimator_, X_train, y_train)
save_result(hgb_tuned_row, REPORTS_DIR / 'model_comparison.csv')
print(hgb_search.best_params_)
hgb_tuned_row

{'model__min_samples_leaf': 50, 'model__max_leaf_nodes': 31, 'model__max_depth': None, 'model__learning_rate': 0.1, 'model__l2_regularization': 10.0}


{'name': 'HistGradientBoosting-tuned',
 'pr_auc_mean': np.float64(0.4489031385914375),
 'pr_auc_std': np.float64(0.011179245171381661),
 'roc_auc_mean': np.float64(0.7977679267814354),
 'roc_auc_std': np.float64(0.0008955211949887608),
 'f1_mean': np.float64(0.44629131359350793),
 'f1_std': np.float64(0.0072685904741842935),
 'precision_mean': np.float64(0.34740152771926525),
 'precision_std': np.float64(0.01008253642039383),
 'recall_mean': np.float64(0.6244394762770005),
 'recall_std': np.float64(0.007022697313936853),
 'train_pr_auc_mean': np.float64(0.5195690127512929),
 'train_pr_auc_std': np.float64(0.006918019124769132),
 'fit_time_mean': np.float64(0.7524868965148925)}

### Observation and decision
The search uses the pipeline's `model__` prefix because the classifier is the `model` step inside `build()`. `average_precision` is the primary score because the subscribed class is uncommon; the other metrics from `cv_report` remain useful supporting evidence.

## k-nearest neighbours baseline and tuning

In [12]:
# The shared pipeline scales numeric columns before kNN calculates distances.
knn_baseline = build(KNeighborsClassifier(n_neighbors=15, n_jobs=-1))
knn_baseline_row = cv_report('KNeighbors', knn_baseline, X_train, y_train)
save_result(knn_baseline_row, REPORTS_DIR / 'model_comparison.csv')
knn_baseline_row

{'name': 'KNeighbors',
 'pr_auc_mean': np.float64(0.33552064577746576),
 'pr_auc_std': np.float64(0.01174761637583533),
 'roc_auc_mean': np.float64(0.7291475203855334),
 'roc_auc_std': np.float64(0.0054706950846841315),
 'f1_mean': np.float64(0.2564344507180429),
 'f1_std': np.float64(0.011336178847175965),
 'precision_mean': np.float64(0.6075185327033998),
 'precision_std': np.float64(0.029026018690497674),
 'recall_mean': np.float64(0.1626092368838984),
 'recall_std': np.float64(0.007966076905706814),
 'train_pr_auc_mean': np.float64(0.4512030934814706),
 'train_pr_auc_std': np.float64(0.0029857157330334403),
 'fit_time_mean': np.float64(0.08607382774353027)}

### Observation and decision
kNN predicts from distances to nearby training examples. Scaling the numeric features in the pipeline matters because a large-scale column such as balance could otherwise dominate the distance. kNN can also be slow at prediction time: for every new client it compares that row with many stored training rows instead of using a compact fitted formula.

In [13]:
knn_search = GridSearchCV(
    estimator=build(KNeighborsClassifier(n_jobs=-1)),
    param_grid={
        'model__n_neighbors': [5, 15, 31, 51],
        'model__weights': ['uniform', 'distance'],
        'model__p': [1, 2],
    },
    scoring='average_precision',
    cv=cv,
    n_jobs=-1,
)
knn_search.fit(X_train, y_train)
knn_tuned_row = cv_report('KNeighbors-tuned', knn_search.best_estimator_, X_train, y_train)
save_result(knn_tuned_row, REPORTS_DIR / 'model_comparison.csv')
print(knn_search.best_params_)
knn_tuned_row

{'model__n_neighbors': 51, 'model__p': 1, 'model__weights': 'distance'}


{'name': 'KNeighbors-tuned',
 'pr_auc_mean': np.float64(0.3887053241589263),
 'pr_auc_std': np.float64(0.010675130269362683),
 'roc_auc_mean': np.float64(0.754646074783374),
 'roc_auc_std': np.float64(0.0012793498224640777),
 'f1_mean': np.float64(0.23764979939351485),
 'f1_std': np.float64(0.011928189933326777),
 'precision_mean': np.float64(0.671732751188115),
 'precision_std': np.float64(0.030047119071883486),
 'recall_mean': np.float64(0.14440899740706317),
 'recall_std': np.float64(0.007865247739164657),
 'train_pr_auc_mean': np.float64(1.0),
 'train_pr_auc_std': np.float64(0.0),
 'fit_time_mean': np.float64(0.07604861259460449)}

### Observation and decision
The grid compares neighbourhood size, distance weighting, and Manhattan versus Euclidean distance. A small neighbourhood can be sensitive to noise, while a large one can wash out local patterns; cross-validation chooses the trade-off instead of guessing.

## Preserve the best-parameter record

In [14]:
# Update the JSON object so teammate entries are retained.
if BEST_PARAMS_PATH.exists():
    best_params = json.loads(BEST_PARAMS_PATH.read_text())
else:
    best_params = {}
best_params['HistGradientBoosting-tuned'] = hgb_search.best_params_
best_params['KNeighbors-tuned'] = knn_search.best_params_
BEST_PARAMS_PATH.write_text(json.dumps(best_params, indent=2) + '\n')
print(json.dumps(best_params, indent=2))

{
  "DecisionTree-tuned": {
    "model__criterion": "gini",
    "model__max_depth": null,
    "model__min_samples_leaf": 50
  },
  "RandomForest-tuned": {
    "model__n_estimators": 500,
    "model__min_samples_leaf": 10,
    "model__max_features": 0.5,
    "model__max_depth": 20
  },
  "HistGradientBoosting-tuned": {
    "model__min_samples_leaf": 50,
    "model__max_leaf_nodes": 31,
    "model__max_depth": null,
    "model__learning_rate": 0.1,
    "model__l2_regularization": 10.0
  },
  "KNeighbors-tuned": {
    "model__n_neighbors": 51,
    "model__p": 1,
    "model__weights": "distance"
  }
}


### Observation and decision
The baseline and tuned rows are saved with distinct names, so the comparison report can show whether tuning helped. The best-parameter file is loaded, updated, and written back, preserving the existing DecisionTree and RandomForest entries.

## Final student-level takeaway

HistGradientBoosting is scikit-learn's fast implementation of gradient boosting, the GBM idea from our lectures: later trees concentrate on errors left by earlier trees. kNN is easy to understand, but it needs scaled numeric inputs because distance is its whole decision rule, and it can be slow when predicting because it searches through the stored training examples. We will choose between the tuned candidates using cross-validated PR-AUC first, then use recall, precision, and ROC-AUC to understand the practical trade-off.